# CarDD Data Analysis — Exploratory Data Analysis (EDA)

**Pipeline position:** 01 — Data understanding, quality assessment, and exploratory analysis.

This notebook analyzes the original CarDD COCO dataset **before annotation conversion and model training**.

It covers:
- dataset composition and split balance
- class imbalance
- image-level object density
- image dimensions and aspect ratio
- bounding-box size and shape
- spatial distribution of damage
- annotation quality
- train / validation / test consistency
- representative visual inspection

> COCO → YOLO conversion is intentionally kept in Notebook 02 to avoid duplicated preprocessing.


## 1. Imports

In [ ]:
from pathlib import Path
import json
import zipfile
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display


## 2. Dataset Path

Update `CARDD_ZIP` if necessary. The source ZIP is read without modifying it.


In [ ]:
CARDD_ZIP = Path(r"C:\Users\Xlosn\Downloads\CarDD_release.zip")
WORKING_DIR = Path.cwd()

if not CARDD_ZIP.exists():
    raise FileNotFoundError(
        f"CarDD ZIP was not found: {CARDD_ZIP}\n"
        "Update CARDD_ZIP to the correct location."
    )

print("Dataset ZIP:", CARDD_ZIP)


## 3. Inspect Dataset Structure

In [ ]:
with zipfile.ZipFile(CARDD_ZIP, "r") as z:
    names = z.namelist()

json_files = [n for n in names if n.lower().endswith(".json")]
image_files = [
    n for n in names
    if n.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
]

print(f"Total files in ZIP: {len(names):,}")
print(f"JSON files: {len(json_files):,}")
print(f"Image files: {len(image_files):,}")

print("\nJSON files:")
for name in json_files:
    print(" -", name)


## 4. Load COCO Annotation Files

In [ ]:
with zipfile.ZipFile(CARDD_ZIP, "r") as z:
    annotations = {}

    for split in ["train", "val", "test"]:
        matches = [
            n for n in json_files
            if f"instances_{split}2017.json" in n.lower()
        ]
        if not matches:
            matches = [n for n in json_files if split in Path(n).stem.lower()]

        if matches:
            with z.open(matches[0]) as f:
                annotations[split] = json.load(f)

for split, data in annotations.items():
    print(
        f"{split:>5}: "
        f"{len(data.get('images', [])):>5,} images | "
        f"{len(data.get('annotations', [])):>6,} annotations | "
        f"{len(data.get('categories', [])):>2,} categories"
    )


## 5. Build Analysis Tables

In [ ]:
image_rows = []
annotation_rows = []
category_rows = []

for split, data in annotations.items():
    cat_map = {c["id"]: c.get("name", str(c["id"])) for c in data.get("categories", [])}

    for c in data.get("categories", []):
        category_rows.append({
            "split": split,
            "category_id": c["id"],
            "category_name": c.get("name", str(c["id"]))
        })

    for img in data.get("images", []):
        image_rows.append({
            "split": split,
            "image_id": img["id"],
            "file_name": img.get("file_name"),
            "width": img.get("width"),
            "height": img.get("height")
        })

    for ann in data.get("annotations", []):
        row = {
            "split": split,
            "annotation_id": ann.get("id"),
            "image_id": ann.get("image_id"),
            "category_id": ann.get("category_id"),
            "category_name": cat_map.get(ann.get("category_id"), "UNKNOWN"),
            "bbox_x": np.nan,
            "bbox_y": np.nan,
            "bbox_width": np.nan,
            "bbox_height": np.nan,
            "area": ann.get("area", np.nan),
            "iscrowd": ann.get("iscrowd", 0),
            "segmentation": ann.get("segmentation")
        }

        bbox = ann.get("bbox")
        if isinstance(bbox, (list, tuple)) and len(bbox) == 4:
            row["bbox_x"], row["bbox_y"], row["bbox_width"], row["bbox_height"] = bbox

        annotation_rows.append(row)

images_df = pd.DataFrame(image_rows)
annotations_df = pd.DataFrame(annotation_rows)
categories_df = pd.DataFrame(category_rows)

print("Images:", images_df.shape)
print("Annotations:", annotations_df.shape)
print("Categories:", categories_df.shape)

display(images_df.head())
display(annotations_df.drop(columns=["segmentation"], errors="ignore").head())


## 6. Dataset Overview

In [ ]:
split_summary = (
    images_df.groupby("split")
    .agg(
        images=("image_id", "nunique"),
        mean_width=("width", "mean"),
        mean_height=("height", "mean"),
    )
    .join(
        annotations_df.groupby("split")["annotation_id"].nunique().rename("annotations")
    )
)

split_summary["annotations_per_image"] = (
    split_summary["annotations"] / split_summary["images"]
)

display(split_summary.round(2))

print(f"Total images:      {images_df['image_id'].nunique():,}")
print(f"Total annotations: {annotations_df['annotation_id'].nunique():,}")
print(f"Total classes:     {categories_df['category_name'].nunique():,}")


## 7. Class Distribution and Imbalance

We analyze both annotation frequency and the number of unique images containing each damage class.


In [ ]:
class_counts = (
    annotations_df["category_name"]
    .value_counts()
    .rename("annotation_count")
    .to_frame()
)

class_counts["percentage"] = (
    class_counts["annotation_count"] / class_counts["annotation_count"].sum() * 100
)

display(class_counts.round(2))

plt.figure(figsize=(11, 6))
class_counts["annotation_count"].sort_values().plot(kind="barh")
plt.xlabel("Number of annotations")
plt.ylabel("Damage class")
plt.title("Damage Class Distribution")
plt.tight_layout()
plt.show()

images_per_class = (
    annotations_df.groupby("category_name")["image_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("unique_images")
    .to_frame()
)

images_per_class["percentage_of_dataset_images"] = (
    images_per_class["unique_images"] /
    images_df["image_id"].nunique() * 100
)

display(images_per_class.round(2))


## 8. Class Distribution by Split

In [ ]:
class_split = pd.crosstab(
    annotations_df["category_name"],
    annotations_df["split"]
)

class_split_pct = class_split.div(class_split.sum(axis=0), axis=1) * 100

display(class_split)
display(class_split_pct.round(2))

ax = class_split.plot(kind="bar", figsize=(12, 6))
ax.set_xlabel("Damage class")
ax.set_ylabel("Number of annotations")
ax.set_title("Class Distribution Across Splits")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Objects per Image

Annotation count alone does not describe image complexity. We therefore calculate how many damage instances occur in each image, including images with zero annotations if they exist.


In [ ]:
object_counts = (
    annotations_df.groupby(["split", "image_id"])
    .size()
    .rename("objects_per_image")
    .reset_index()
)

all_image_keys = images_df[["split", "image_id"]].drop_duplicates()

object_counts = all_image_keys.merge(
    object_counts,
    on=["split", "image_id"],
    how="left"
)

object_counts["objects_per_image"] = (
    object_counts["objects_per_image"].fillna(0).astype(int)
)

display(
    object_counts.groupby("split")["objects_per_image"]
    .describe()
    .round(2)
)

multi_object = (
    object_counts.assign(
        multiple_damage=object_counts["objects_per_image"] > 1,
        no_damage=object_counts["objects_per_image"] == 0
    )
    .groupby("split")
    .agg(
        images=("image_id", "size"),
        zero_object_images=("no_damage", "sum"),
        multi_object_images=("multiple_damage", "sum")
    )
)

multi_object["multi_object_percentage"] = (
    multi_object["multi_object_images"] /
    multi_object["images"] * 100
)

display(multi_object.round(2))

plt.figure(figsize=(10, 6))
object_counts["objects_per_image"].value_counts().sort_index().head(20).plot(kind="bar")
plt.xlabel("Number of objects in an image")
plt.ylabel("Number of images")
plt.title("Distribution of Damage Instances per Image")
plt.tight_layout()
plt.show()


## 10. Image Dimensions and Aspect Ratio

In [ ]:
images_df["aspect_ratio"] = images_df["width"] / images_df["height"]
images_df["image_area"] = images_df["width"] * images_df["height"]

display(
    images_df[["width", "height", "aspect_ratio", "image_area"]]
    .describe()
    .round(2)
)

display(
    images_df.groupby("split")[["width", "height", "aspect_ratio"]]
    .agg(["mean", "median", "min", "max"])
    .round(2)
)

plt.figure(figsize=(10, 6))
plt.hist(images_df["aspect_ratio"].dropna(), bins=40)
plt.xlabel("Width / Height")
plt.ylabel("Number of images")
plt.title("Image Aspect Ratio Distribution")
plt.tight_layout()
plt.show()


## 11. Bounding-Box Size and Shape

We examine absolute and normalized damage size:
- width and height
- normalized width and height
- box area
- box area relative to image area
- box aspect ratio


In [ ]:
bbox_df = annotations_df.merge(
    images_df[["split", "image_id", "width", "height"]],
    on=["split", "image_id"],
    how="left"
)

bbox_df["bbox_right"] = bbox_df["bbox_x"] + bbox_df["bbox_width"]
bbox_df["bbox_bottom"] = bbox_df["bbox_y"] + bbox_df["bbox_height"]

bbox_df["bbox_width_norm"] = bbox_df["bbox_width"] / bbox_df["width"]
bbox_df["bbox_height_norm"] = bbox_df["bbox_height"] / bbox_df["height"]
bbox_df["bbox_area_calc"] = bbox_df["bbox_width"] * bbox_df["bbox_height"]

bbox_df["bbox_area_ratio"] = (
    bbox_df["bbox_area_calc"] /
    (bbox_df["width"] * bbox_df["height"])
)

bbox_df["bbox_aspect_ratio"] = (
    bbox_df["bbox_width"] /
    bbox_df["bbox_height"].replace(0, np.nan)
)

display(
    bbox_df[
        [
            "bbox_width", "bbox_height",
            "bbox_width_norm", "bbox_height_norm",
            "bbox_area_calc", "bbox_area_ratio",
            "bbox_aspect_ratio"
        ]
    ].describe().round(4)
)

display(
    bbox_df.groupby("category_name")[
        ["bbox_width", "bbox_height", "bbox_area_ratio", "bbox_aspect_ratio"]
    ]
    .median()
    .sort_values("bbox_area_ratio", ascending=False)
    .round(4)
)


### 11.1 Bounding-Box Area Distribution

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(bbox_df["bbox_area_ratio"].dropna(), bins=50)
plt.xlabel("Bounding-box area / image area")
plt.ylabel("Number of annotations")
plt.title("Relative Damage Size Distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
bbox_df.boxplot(
    column="bbox_area_ratio",
    by="category_name",
    rot=45
)
plt.suptitle("")
plt.title("Relative Bounding-Box Area by Damage Class")
plt.xlabel("Damage class")
plt.ylabel("Relative box area")
plt.tight_layout()
plt.show()


## 12. Spatial Distribution of Damage

Bounding-box centers are normalized to `[0, 1]` so different image resolutions can be compared.


In [ ]:
bbox_df["center_x_norm"] = (
    bbox_df["bbox_x"] + bbox_df["bbox_width"] / 2
) / bbox_df["width"]

bbox_df["center_y_norm"] = (
    bbox_df["bbox_y"] + bbox_df["bbox_height"] / 2
) / bbox_df["height"]

display(
    bbox_df[["center_x_norm", "center_y_norm"]]
    .describe()
    .round(3)
)

plt.figure(figsize=(9, 7))
plt.hist2d(
    bbox_df["center_x_norm"].dropna(),
    bbox_df["center_y_norm"].dropna(),
    bins=30
)
plt.xlabel("Normalized damage center X")
plt.ylabel("Normalized damage center Y")
plt.title("Spatial Distribution of Damage Centers")
plt.colorbar(label="Annotation count")
plt.tight_layout()
plt.show()


### 12.1 Spatial Distribution by Class

In [ ]:
for class_name in sorted(bbox_df["category_name"].dropna().unique()):
    subset = bbox_df[bbox_df["category_name"] == class_name]

    plt.figure(figsize=(7, 5))
    plt.scatter(
        subset["center_x_norm"],
        subset["center_y_norm"],
        s=10,
        alpha=0.25
    )
    plt.xlim(0, 1)
    plt.ylim(1, 0)
    plt.xlabel("Normalized center X")
    plt.ylabel("Normalized center Y")
    plt.title(f"Damage Location — {class_name}")
    plt.tight_layout()
    plt.show()


## 13. Annotation Quality Checks

Checks include:
- missing image dimensions
- annotations referencing missing images
- invalid category IDs
- non-positive bounding boxes
- bounding boxes outside image boundaries
- crowd annotations
- missing/empty segmentation
- duplicate annotation IDs
- duplicate annotation records


In [ ]:
quality_results = {}

quality_results["missing_width"] = int(images_df["width"].isna().sum())
quality_results["missing_height"] = int(images_df["height"].isna().sum())

image_keys = set(zip(images_df["split"], images_df["image_id"]))
ann_keys = set(zip(annotations_df["split"], annotations_df["image_id"]))

quality_results["annotations_with_missing_images"] = len(ann_keys - image_keys)

valid_category_ids = set(categories_df["category_id"])
quality_results["invalid_category_ids"] = int(
    (~annotations_df["category_id"].isin(valid_category_ids)).sum()
)

quality_results["non_positive_bbox_width"] = int(
    (bbox_df["bbox_width"] <= 0).sum()
)
quality_results["non_positive_bbox_height"] = int(
    (bbox_df["bbox_height"] <= 0).sum()
)

quality_results["bbox_outside_left"] = int((bbox_df["bbox_x"] < 0).sum())
quality_results["bbox_outside_top"] = int((bbox_df["bbox_y"] < 0).sum())
quality_results["bbox_outside_right"] = int(
    (bbox_df["bbox_right"] > bbox_df["width"]).sum()
)
quality_results["bbox_outside_bottom"] = int(
    (bbox_df["bbox_bottom"] > bbox_df["height"]).sum()
)

quality_results["iscrowd_annotations"] = int(
    (annotations_df["iscrowd"] == 1).sum()
)

quality_results["missing_or_empty_segmentation"] = int(
    annotations_df["segmentation"].apply(
        lambda x: x is None or x == [] or x == {}
    ).sum()
)

quality_results["duplicate_annotation_ids"] = int(
    annotations_df.duplicated(["split", "annotation_id"]).sum()
)

quality_results["duplicate_annotation_records"] = int(
    bbox_df.duplicated(
        [
            "split", "image_id", "category_id",
            "bbox_x", "bbox_y", "bbox_width", "bbox_height"
        ]
    ).sum()
)

quality_table = (
    pd.Series(quality_results, name="count")
    .to_frame()
    .sort_values("count", ascending=False)
)

display(quality_table)


## 14. Train / Validation / Test Consistency

In [ ]:
split_comparison = (
    object_counts.groupby("split")["objects_per_image"]
    .agg(
        images="count",
        mean_objects="mean",
        median_objects="median",
        max_objects="max"
    )
)

split_comparison["annotations"] = (
    annotations_df.groupby("split")["annotation_id"].nunique()
)

split_comparison["annotations_per_image"] = (
    split_comparison["annotations"] / split_comparison["images"]
)

display(split_comparison.round(3))

split_class_pct = pd.crosstab(
    annotations_df["split"],
    annotations_df["category_name"],
    normalize="index"
) * 100

display(split_class_pct.round(2))


## 15. Visual Inspection of Representative Samples

We extract a small number of images from the ZIP only for inspection and draw their COCO bounding boxes.

This helps detect systematic annotation problems that numerical checks cannot reveal.


In [ ]:
SAMPLE_DIR = WORKING_DIR / "cardd_eda_samples"
SAMPLE_DIR.mkdir(exist_ok=True)

def find_image_member(file_name, split):
    base = Path(file_name).name

    candidates = [
        n for n in image_files
        if Path(n).name == base and split in n.lower()
    ]

    if not candidates:
        candidates = [n for n in image_files if Path(n).name == base]

    return candidates[0] if candidates else None

def draw_sample(split, image_id):
    data = annotations[split]

    image_record = next(
        img for img in data["images"]
        if img["id"] == image_id
    )

    member = find_image_member(image_record["file_name"], split)

    if member is None:
        print("Image not found:", image_record["file_name"])
        return

    output_path = SAMPLE_DIR / Path(image_record["file_name"]).name

    with zipfile.ZipFile(CARDD_ZIP, "r") as z:
        with z.open(member) as src, open(output_path, "wb") as dst:
            dst.write(src.read())

    image = Image.open(output_path).convert("RGB")

    plt.figure(figsize=(10, 7))
    plt.imshow(image)

    cat_map = {
        c["id"]: c.get("name", str(c["id"]))
        for c in data["categories"]
    }

    for ann in data["annotations"]:
        if ann["image_id"] != image_id:
            continue

        x, y, w, h = ann["bbox"]

        rect = plt.Rectangle(
            (x, y), w, h,
            fill=False,
            linewidth=2
        )

        plt.gca().add_patch(rect)

        plt.text(
            x,
            max(0, y - 4),
            cat_map.get(ann["category_id"], "UNKNOWN"),
            fontsize=9
        )

    plt.title(f"{split.upper()} — {image_record['file_name']}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

for split in annotations:
    candidates = (
        object_counts[
            (object_counts["split"] == split) &
            (object_counts["objects_per_image"] > 1)
        ]
        .sort_values("objects_per_image", ascending=False)
        .head(1)
    )

    if not candidates.empty:
        draw_sample(split, int(candidates.iloc[0]["image_id"]))


## 16. EDA Summary

In [ ]:
largest_class = class_counts.index[0]

print("=== CarDD EDA Summary ===")
print(f"Images: {len(images_df):,}")
print(f"Annotations: {len(annotations_df):,}")
print(f"Classes: {annotations_df['category_name'].nunique():,}")
print(
    f"Most frequent class: {largest_class} "
    f"({class_counts.iloc[0]['annotation_count']:,} annotations, "
    f"{class_counts.iloc[0]['percentage']:.2f}%)"
)
print(
    f"Median objects/image: "
    f"{object_counts['objects_per_image'].median():.2f}"
)
print(
    f"Median relative bbox area: "
    f"{bbox_df['bbox_area_ratio'].median():.4f}"
)

print("\nNon-zero data-quality findings:")
non_zero = quality_table[quality_table["count"] > 0]

if non_zero.empty:
    print("No issues detected by the checks above.")
else:
    display(non_zero)


## 17. Handoff to Notebook 02

Notebook 01 ends with **data understanding and quality assessment**.

Next:

**02 — Annotation Conversion**
- COCO → YOLO segmentation
- category ID mapping
- polygon validation
- dataset structure
- `cardd.yaml`
- converted-label validation

Then:

**03 — Baseline Model**
- load the processed YOLO dataset
- train the segmentation baseline
- validate the model
- inspect results

The notebooks are intentionally separated so preprocessing logic is not duplicated.
